# Chapter 4: Implementing a GPT Model from Scratch

In this notebook, we:
1. Instantiate a GPT-2 Small model from scratch
2. Inspect the architecture and count parameters
3. Run a forward pass with random tokens
4. Generate text (random weights → random output, just verifying the pipeline)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
torch.manual_seed(42)

## 1. Configuration

GPT-2 Small: 12 layers, 12 heads, 768 embedding dim, 50,257 vocab.

In [ ]:
from ch04.config import GPT2_SMALL

cfg = GPT2_SMALL
print(f"vocab_size:   {cfg.vocab_size:,}")
print(f"d_model:      {cfg.d_model}")
print(f"n_heads:      {cfg.n_heads}")
print(f"n_layers:     {cfg.n_layers}")
print(f"max_seq_len:  {cfg.max_seq_len}")
print(f"head_dim:     {cfg.d_model // cfg.n_heads}")

## 2. Instantiate the Model

In [ ]:
from ch04.gpt_model import GPTModel

model = GPTModel(cfg)

total_params = model.count_parameters()
print(f"Total trainable parameters: {total_params:,}")
print(f"Approximate size in memory: {total_params * 4 / 1e6:.1f} MB (float32)")

In [ ]:
# Detailed parameter breakdown
print("Parameter breakdown:")
print("=" * 60)
for name, param in model.named_parameters():
    if 'blocks.0' in name or 'blocks' not in name:
        suffix = " (× 12 blocks)" if name == 'blocks.0.ln1.gamma' else ""
        print(f"  {name:45s} {str(list(param.shape)):20s} = {param.numel():>12,}{suffix}")
    elif 'blocks.1.ln1.gamma' in name:
        print(f"  ... (blocks 1-11 same as block 0)")

## 3. Forward Pass

Feed random token IDs through the model and inspect the output shape.

In [ ]:
# Create random input: batch=2, seq_len=16
batch_size, seq_len = 2, 16
idx = torch.randint(0, cfg.vocab_size, (batch_size, seq_len))
print(f"Input shape:  {idx.shape}")
print(f"Input tokens: {idx[0].tolist()[:8]}...")

# Forward pass
with torch.no_grad():
    logits = model(idx)

print(f"Output shape: {logits.shape}")  # (2, 16, 50257)
print(f"Logits range: [{logits.min():.3f}, {logits.max():.3f}]")

In [ ]:
# Check that softmax over logits gives a valid probability distribution
probs = torch.softmax(logits[0, -1, :], dim=-1)
print(f"Probabilities sum:  {probs.sum():.6f}")  # Should be ~1.0
print(f"Max probability:    {probs.max():.6f}")
print(f"Predicted token ID: {probs.argmax().item()}")

## 4. Individual Components

Test each building block independently.

In [ ]:
from ch04.gpt_model import LayerNorm, GELU, FeedForward, TransformerBlock

x = torch.randn(2, 8, cfg.d_model)  # (batch=2, seq_len=8, d_model=768)

# LayerNorm
ln = LayerNorm(cfg.d_model)
out = ln(x)
print(f"LayerNorm:   {x.shape} → {out.shape}")
print(f"  Mean ≈ 0: {out[0, 0].mean():.6f}")
print(f"  Std  ≈ 1: {out[0, 0].std():.6f}")

# GELU
gelu = GELU()
test_vals = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"\nGELU({test_vals.tolist()})")
print(f"  = {gelu(test_vals).tolist()}")

# FeedForward
ff = FeedForward(cfg.d_model, dropout=0.0)
out = ff(x)
print(f"\nFeedForward: {x.shape} → {out.shape}")
ff_params = sum(p.numel() for p in ff.parameters())
print(f"  FF params: {ff_params:,} (768→3072→768)")

# TransformerBlock
block = TransformerBlock(cfg)
out = block(x)
print(f"\nTransformerBlock: {x.shape} → {out.shape}")
block_params = sum(p.numel() for p in block.parameters())
print(f"  Block params: {block_params:,}")

## 5. Text Generation

The model has random weights, so the output will be random tokens.
This just validates that the generation pipeline works correctly.

In [ ]:
from ch04.generate import generate_greedy, generate_topk

# Use a small prompt (just a few token IDs)
prompt = torch.tensor([[15496, 11, 314, 716]])  # "Hello, I am" in GPT-2 tokens
print(f"Prompt tokens: {prompt.tolist()}")
print(f"Prompt length: {prompt.shape[1]}")

In [ ]:
# Greedy decoding
print("--- Greedy Decoding ---")
generated = generate_greedy(model, prompt, max_new_tokens=20)
print(f"Generated tokens: {generated[0].tolist()}")
print(f"New tokens:       {generated[0, prompt.shape[1]:].tolist()}")

In [ ]:
# Top-k sampling with different temperatures
print("--- Top-k Sampling ---")
for temp in [0.5, 1.0, 2.0]:
    gen = generate_topk(model, prompt, max_new_tokens=20, temperature=temp, top_k=10)
    new_tokens = gen[0, prompt.shape[1]:].tolist()
    print(f"  T={temp:.1f}, k=10: {new_tokens}")

In [ ]:
# Verify greedy is deterministic (same output every time)
print("--- Greedy Determinism Check ---")
gen1 = generate_greedy(model, prompt, max_new_tokens=10)
gen2 = generate_greedy(model, prompt, max_new_tokens=10)
print(f"Run 1: {gen1[0].tolist()}")
print(f"Run 2: {gen2[0].tolist()}")
print(f"Identical: {torch.equal(gen1, gen2)}")

## Summary

✅ Built the complete GPT architecture from scratch:
- **LayerNorm**: Manual implementation (no `nn.LayerNorm`)
- **GELU**: Smooth activation used in GPT (not ReLU)
- **FeedForward**: Linear → GELU → Linear → Dropout (inner dim = 4×d_model)
- **TransformerBlock**: Pre-norm with residual connections
- **GPTModel**: Embeddings → N blocks → LayerNorm → Linear head

✅ Verified with GPT-2 Small config (~163M params untied)

✅ Generation works (greedy + top-k sampling) — output is random since weights are untrained

**Next**: Chapter 5 — Pretraining on Unlabeled Data